# LFG公开公式 Post-hoc K=0：阶段1固定消息通道
本包只做Wan终态一次正交3D Haar LLL扩频、相关符号和BCH。不是时间ID/局部化/状态空间或真实多步同步。没有作者源码，仅称公开公式参考实现，不宣称论文等价。原用户净32位仍另读退步，不把新固定432位诊断消息叫增容。
四个已见pre-tag P0来源、181frames320x512。只选alpha .15一次；W除sqrt512，LLL每channel population sigma。全部16channel和旧guards参与，不能套旧QIM保护掩码；保存actual FP32扰动与7高子带roundoff，不加幅/扫描/投影补偿。BCH母码1023/943/t8 GF0x409缩短511信息0→512/432，全MSB-first，是参考补定。
P0/MSG同runtime新D→RGB8；未压缩RGB8与同份RGB8的H264 CRF23 MP4分别E。VAE1/D8/codec各8/E16、FFT16供两钥32/Counter12288；48消息盲表，原32位固定256group/2816cell配对；16质量+8clipping。无float-clean E/裁片/phase/path/新DiT/梯度/训练。
先保存并回读全部scores/raw512、BCH decoded432与原payload，再开真值。BCH_SUCCESS不是完整消息正确或认证；>8可误纠，失败432位保留未决、不偷用raw fallback。对照输出和全部失败保留，四来源非独立确认。
已完成必要CPU/合成/stub核验，并按用户授权发布到GitHub现有dev/local-joint-state-payload-v1分支。未真实运行模型、GPU、媒体或Colab，验证由用户Run all执行。直接下载可用源码，无巨量SOURCE_TEXT/B64/hash/manifest/版本/Git-clean门禁或手工ZIP。
一次attempt/每call STARTED先持久化，失败/中断消费，后续Run all只读已有JSON，不重试/续算/fallback。GPU由用户运行。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, subprocess
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-LFG-Posthoc-Reference-V1/fixed-viewed-stage1-v1')
def execution_artifacts(output):
    out=Path(output);found=[]
    for name in ('attempt.json','ledger.json','result.json','evaluation.json','plan.json','fixed_rows.json','progress.json','post_truth_access.json','message_post_rows.json','payload_group_pairs.json','payload_cell_pairs.json'):
        for suffix in ('','.tmp'):
            path=out/(name+suffix)
            if path.exists() or path.is_symlink():found.append(name+suffix)
    for name in ('raw','blind','payload','private','PN','construction','native','latents','media','quality','markers','receipts'):
        root=out/name
        if root.exists():found.extend(str(x.relative_to(out)) for x in root.rglob('*') if x.is_file() or x.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    SOURCE_ROOT=Path('/content/SC-SSTW-b-lfg-posthoc-reference-v1')
    if not SOURCE_ROOT.exists():
        subprocess.run(['git','clone','--depth','1','--single-branch','--branch','dev/local-joint-state-payload-v1','https://github.com/RICHAAARC/SC-SSTW.git',str(SOURCE_ROOT)],check=True)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/b_lfg_posthoc_reference_v1_run.py').is_file():
        raise FileNotFoundError('Available source lacks the LFG reference entry: '+str(SOURCE_ROOT))
    OUTPUT.mkdir(parents=True,exist_ok=True)
else:
    print('Existing scientific artifacts: source/setup/model/physical execution skipped; saved JSON read-only.')


固定运行。缺依赖只修真实导入/ffmpeg故障；生产不依赖bchlib，也不运行CPU oracle。

In [ ]:

import subprocess, sys, time
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','update'], 'refresh package metadata for missing FFmpeg')
        command(['apt-get','install','-y','ffmpeg'], 'install missing ffmpeg/ffprobe')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe unavailable after dependency installation; see '+str(LOG))
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan VAE dependencies')
    with (OUTPUT/'runner.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'experiments.wan_state_clock.b_lfg_posthoc_reference_v1_run', '--output', str(OUTPUT)],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    try:
        while process.poll() is None:
            time.sleep(30)
            try:
                progress = json.loads((OUTPUT/'progress.json').read_text())
                print('Units', progress.get('counts',{}).get('RETURNED',0), '/', progress['planned'], 'counts', progress.get('counts',{}), 'active', progress.get('active'), flush=True)
            except (OSError,ValueError,KeyError,TypeError) as error:
                print('Progress read unavailable; observing same worker:',type(error).__name__,str(error),flush=True)
    except KeyboardInterrupt:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
        print('Stopped; saved attempts consumed. Later Run all is read-only.')
        raise
    print('Runner return code',process.returncode,'; inspect saved results and runner.log. No retry.')
else:
    print('Existing attempt: source/setup/physical execution skipped.')


保存结果只读：raw coded BER、BCH状态、完整432消息和原32位退步分开。

In [ ]:

for name in ('result.json','evaluation.json'):
    path=OUTPUT/name
    if not path.is_file():print('Unavailable',name);continue
    try:value=json.loads(path.read_text())
    except (OSError,ValueError) as e:print('Saved read error',name,str(e));continue
    if name=='result.json':print('Execution',value.get('status'),'physical calls',value.get('kinds_started'),'fixed',value.get('fixed_denominators'))
    else:
        for group,stats in value.get('message_summary',{}).items():print('Message',group,stats)
        print('Original net32 payload',value.get('payload_by_view'),'reference issues',value.get('reference_issues'))
print('Actual native/LLL/FP32/guards/user-channel costs: construction/. Global quality/temporal/clipping: quality/.')
print('BCH success is not exact recovery or authentication; failed decoded bits remain unresolved. No local/time-ID/sync or capacity claim. Output:',OUTPUT)
